In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pandas as pd
from pathlib import Path
from datetime import date

input_workbook = ""
file_2023 = "/content/drive/MyDrive/dataset/Datos finales 2023.xlsx"
file_2024 = "/content/drive/MyDrive/dataset/Datos finales 2024.xlsx"
sheets_to_process = ["SE", "CE", "SO2", "SUR"]  # sheets a procesar
output_workbook = "datos_modificados_final.xlsx"

def season_astronomical_mexico(dt):

    if pd.isna(dt):
        return None
    # convertir a date si es Timestamp
    if hasattr(dt, "date"):
        d = dt.date()
    else:
        d = dt  # ya es date
    y = d.year
    # definimos los inicios para el mismo año
    spring_start = date(y, 3, 20)
    summer_start = date(y, 6, 21)
    autumn_start = date(y, 9, 23)
    winter_start = date(y, 12, 22)
    if d >= winter_start:
        return "invierno"
    if d >= autumn_start:
        return "otoño"
    if d >= summer_start:
        return "verano"
    if d >= spring_start:
        return "primavera"
    return "invierno"

def safe_parse_date(df, date_col="day"):
    if date_col not in df.columns:
        raise KeyError(f"No se encontró la columna '{date_col}' en el dataframe.")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df

def add_astronomical_season_dummies(df, date_col="day", prefix="is_"):
    df = safe_parse_date(df, date_col)
    if df[date_col].isna().any():
        print("Advertencia: hay valores de 'day' que no se pudieron parsear y serán NaT.")
    # Aplicar la función de estación astronómica
    df["season"] = df[date_col].apply(season_astronomical_mexico)
    # Crear dummies 0/1 y asegurar que existan todas las columnas (aunque falte alguna estación en los datos)
    df[prefix + "primavera"] = (df["season"] == "primavera").astype(int)
    df[prefix + "verano"]    = (df["season"] == "verano").astype(int)
    df[prefix + "otoño"]     = (df["season"] == "otoño").astype(int)
    df[prefix + "invierno"]  = (df["season"] == "invierno").astype(int)
    return df

def process_and_concat_sheet(base_name, input_workbook, file_2023="", file_2024=""):

    parts = []
    if input_workbook and Path(input_workbook).exists():
        wb = pd.ExcelFile(input_workbook, engine="openpyxl")
        for s in wb.sheet_names:
            s_low = s.lower()
            if base_name.lower() in s_low and "2023" in s_low:
                parts.append(("2023", wb.parse(s)))
            if base_name.lower() in s_low and "2024" in s_low:
                parts.append(("2024", wb.parse(s)))
    if not any(p[0]=="2023" for p in parts) and file_2023 and Path(file_2023).exists():
        xls = pd.ExcelFile(file_2023, engine="openpyxl")
        for s in xls.sheet_names:
            if base_name.lower() in s.lower():
                parts.append(("2023", xls.parse(s)))
                break
    if not any(p[0]=="2024" for p in parts) and file_2024 and Path(file_2024).exists():
        xls = pd.ExcelFile(file_2024, engine="openpyxl")
        for s in xls.sheet_names:
            if base_name.lower() in s.lower():
                parts.append(("2024", xls.parse(s)))
                break
    if not parts and input_workbook and Path(input_workbook).exists():
        wb = pd.ExcelFile(input_workbook, engine="openpyxl")
        for y in ("2023","2024"):
            guess = f"{base_name}_{y}"
            if guess in wb.sheet_names:
                parts.append((y, wb.parse(guess)))

    if not parts:
        raise FileNotFoundError(f"No encontré datos para '{base_name}' en los archivos indicados. Revisa nombres de sheets o rutas.")

    # ordenar por año (2023 arriba, 2024 abajo)
    parts_sorted = sorted(parts, key=lambda x: x[0])
    dfs = []
    for yr, df in parts_sorted:
        df = df.copy()
        df["source_year"] = yr
        dfs.append(df)

    concatenated = pd.concat(dfs, ignore_index=True, sort=False)

    # agregar dummies de estaciones astronómicas
    concatenated = add_astronomical_season_dummies(concatenated, date_col="day", prefix="is_")

    # reordenar: mantener 2023 arriba, 2024 abajo, y ordenar por fecha dentro de cada año
    concatenated["source_year_sort"] = concatenated["source_year"].astype(str)
    concatenated = concatenated.sort_values(by=["source_year_sort", "day"], ascending=[True, True]).reset_index(drop=True)
    concatenated = concatenated.drop(columns=["source_year_sort"])

    return concatenated
result_dfs = {}
for base in sheets_to_process:
    try:
        print(f"Procesando {base} ...")
        df_mod = process_and_concat_sheet(base, input_workbook, file_2023=file_2023, file_2024=file_2024)
        result_dfs[base] = df_mod
        print(f"  -> filas: {df_mod.shape[0]}, columnas: {df_mod.shape[1]}")
    except Exception as e:
        print(f"ERROR procesando {base}: {e}")

# Guardar a Excel con cada sheet ya corregida
if result_dfs:
    with pd.ExcelWriter(output_workbook, engine="openpyxl") as writer:
        for sheet_name, df_out in result_dfs.items():
            df_out.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Guardado: '{output_workbook}' con hojas: {list(result_dfs.keys())}")
else:
    print("No se generaron datos para guardar.")


Procesando SE ...
  -> filas: 731, columnas: 22
Procesando CE ...
  -> filas: 731, columnas: 22
Procesando SO2 ...
  -> filas: 731, columnas: 22
Procesando SUR ...
  -> filas: 731, columnas: 22
Guardado: 'datos_modificados_final.xlsx' con hojas: ['SE', 'CE', 'SO2', 'SUR']


In [ ]:
from IPython.display import FileLink

# Crear un link para descargar el archivo
FileLink("datos_modificados_final.xlsx")

/content/datos_modificados_final.xlsx